# 02 Data Cleaning
**Agenda for this notebook:**

- Load the raw dataset
- Clean price column (extract numeric value)
- Clean mileage column
- Handle missing values
- Fix data types (year, price, mileage)
- Remove unrealistic values (e.g. future years)

**Project Notebooks Plan:**

1. 01_data_loading.ipynb ← Completed
2. 02_data_cleaning.ipynb ← Current
3. 03_grouping_aggregation.ipynb
4. 04_feature_engineering.ipynb
5. 05_visualization.ipynb

# Importing Libraries

- We import pandas for data manipulation
- This is the foundation for all cleaning operations

In [1]:
import pandas as pd

# Loading the Raw Dataset

- We always start cleaning by loading fresh raw data
- This keeps our cleaning reproducible
- We work on a copy to protect the original data

In [3]:
# Load the raw dataset
raw_data_path = '../data/raw/pakwheels.csv'   
df = pd.read_csv(raw_data_path)

# Create a working copy for cleaning
df_clean = df.copy()

print(f"Shape: {df_clean.shape}")

Shape: (1394, 18)


# Handling the Price Column

- The price column contains text like "PKR 38.75 lacs"
- We need to extract only the numeric value for machine learning
- This is one of the most important cleaning steps for price prediction

In [4]:
# Create a clean numeric price column
df_clean['price_clean'] = df_clean['price'].astype(str).str.replace('PKR', '', regex=False)
df_clean['price_clean'] = df_clean['price_clean'].str.replace('lacs', '', regex=False)
df_clean['price_clean'] = df_clean['price_clean'].str.strip()

# Convert to float
df_clean['price_clean'] = pd.to_numeric(df_clean['price_clean'], errors='coerce')

print("Price cleaning done!")
print(f"Sample cleaned prices:\n{df_clean[['price', 'price_clean']].head()}")

Price cleaning done!
Sample cleaned prices:
            price  price_clean
0  PKR 38.75 lacs        38.75
1  PKR 40.25 lacs        40.25
2     PKR 35 lacs        35.00
3   PKR 88.5 lacs        88.50
4  PKR 56.85 lacs        56.85


# Handling the Mileage Column

- The mileage column has values like "19,198 km"
- We need to extract only the numeric value (remove "km" and commas)
- This will give us clean mileage in kilometers for modeling

In [5]:
# Clean the mileage column
df_clean['mileage_clean'] = df_clean['mileage'].astype(str).str.replace('km', '', regex=False)
df_clean['mileage_clean'] = df_clean['mileage_clean'].str.replace(',', '', regex=False)
df_clean['mileage_clean'] = df_clean['mileage_clean'].str.strip()

# Convert to numeric
df_clean['mileage_clean'] = pd.to_numeric(df_clean['mileage_clean'], errors='coerce')

print("Mileage cleaning done!")
print(f"Sample cleaned mileage:\n{df_clean[['mileage', 'mileage_clean']].head()}")

Mileage cleaning done!
Sample cleaned mileage:
      mileage  mileage_clean
0   19,198 km        19198.0
1   20,997 km        20997.0
2  180,000 km       180000.0
3   97,000 km        97000.0
4   82,000 km        82000.0


# Dropping Original Messy Columns

- We have created clean versions (price_clean, mileage_clean)
- We can now drop the original price and mileage columns
- This keeps our dataset cleaner and easier to work with

In [10]:
# Safely drop original messy columns if they still exist
columns_to_drop = ['price', 'mileage']
df_clean = df_clean.drop(columns=[col for col in columns_to_drop if col in df_clean.columns])

print(f" Original columns dropped (if existed)!")
print(f"Current shape: {df_clean.shape}")
print("Remaining columns:")
print(df_clean.columns.tolist())

 Original columns dropped (if existed)!
Current shape: (1394, 18)
Remaining columns:
['title', 'url', 'year', 'engine', 'transmission', 'fuel_type', 'city', 'seller_type', 'ad_id', 'posted_date', 'scraped_at', 'source_type', 'source_url', 'make', 'model_slug', 'record_type', 'price_clean', 'mileage_clean']


# Handling Missing Values - Drop Incomplete Rows

- Many rows (63) are almost empty (missing price, title, etc.)
- We will drop these incomplete rows as they have little value for training a model
- This is a standard and safe approach for scraped data

In [11]:
# Drop rows where key columns (price or title) are missing
df_clean = df_clean.dropna(subset=['price_clean', 'title'])

print(f" Incomplete rows removed!")
print(f"Remaining rows: {len(df_clean):,}")

 Incomplete rows removed!
Remaining rows: 1,130


# Checking Remaining Missing Values

- We check how many missing values are left after basic cleaning
- This helps us decide what to do with columns like engine and transmission
- Many missing values are expected in scraped data

In [12]:
# Check missing values after basic cleaning
missing = df_clean.isnull().sum()
print("Remaining Missing Values:")
print(missing[missing > 0])

Remaining Missing Values:
engine           579
transmission     579
make            1130
model_slug      1130
record_type     1130
dtype: int64


# Dropping Completely Empty Columns

- Columns make, model_slug, and record_type have zero useful data|
- We safely drop them as they add no value to price prediction
- This reduces noise in our dataset

In [13]:
# Drop completely empty columns
df_clean = df_clean.drop(['make', 'model_slug', 'record_type'], axis=1)

print(f" Empty columns dropped!")
print(f"Current shape: {df_clean.shape}")
print("Remaining columns:", df_clean.columns.tolist())

 Empty columns dropped!
Current shape: (1130, 15)
Remaining columns: ['title', 'url', 'year', 'engine', 'transmission', 'fuel_type', 'city', 'seller_type', 'ad_id', 'posted_date', 'scraped_at', 'source_type', 'source_url', 'price_clean', 'mileage_clean']


# Handling Remaining Missing Values in Engine & Transmission

- engine and transmission have many missing values (~579)
- For now, we will fill them with "Unknown" so we don't lose rows
- Later we can do more advanced imputation if needed

In [14]:
# Fill missing engine and transmission with 'Unknown'
df_clean['engine'] = df_clean['engine'].fillna('Unknown')
df_clean['transmission'] = df_clean['transmission'].fillna('Unknown')

print(" Missing values handled!")
print("Remaining missing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

 Missing values handled!
Remaining missing values:
Series([], dtype: int64)


# Converting Data Types

- We convert year to integer for proper numerical analysis
- This is important because year is a key feature for predicting car price
- We also check for any unrealistic future years

In [15]:
# Convert year to integer
df_clean['year'] = df_clean['year'].astype(int)

# Check for unrealistic years
future_cars = df_clean[df_clean['year'] > 2025]
print(f"Number of cars with future year (2026+): {len(future_cars)}")

Number of cars with future year (2026+): 125


# Fixing Unrealistic Future Years

- We will replace years greater than 2025 with the median year
- This is a reasonable approach for scraped data with errors
- Preserves the rows while making the data realistic

In [16]:
# Fix future years by replacing with median year
median_year = df_clean['year'].median()
df_clean.loc[df_clean['year'] > 2025, 'year'] = median_year

print(f" Future years fixed using median year: {median_year}")
print(f"New year range: {df_clean['year'].min()} - {df_clean['year'].max()}")

 Future years fixed using median year: 2021.0
New year range: 1981 - 2025


In [ ]:
Final Cleaning Summary

We have completed major cleaning steps
Dataset is now ready with clean numeric price and mileage
This prepares us well for feature engineering and modeling